# Fitting a Linear ACE Potential

This notebook picks up where **SimpleUnary** leaves off: it reads the
training dataset produced there and fits a simple linear pair-descriptor ACE
potential with `python-ace`.

## Prerequisites

Run **SimpleUnary** first so that `Unary/2/everything_training_data.pckl.gz`
exists.  Then install the optional `python-ace` package:

```bash
pip install python-ace
```

## What this notebook covers

1. **Load** the training dataframe from SimpleUnary.
2. **Define** a minimal linear ACE basis (pair descriptors only).
3. **Fit** the potential with ridge regression via `LinearACEFit`.
4. **Evaluate** train errors (MAE / RMSE on energies and forces).
5. **Inspect** the resulting pair potential against the Morse reference.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyace
import pyace.linearacefit as plf
from ase import Atoms
from ase.calculators.morse import MorsePotential

## Load training data

The dataframe was saved by **SimpleUnary** and contains one row per structure
with columns `ase_atoms`, `energy` (eV), `forces` (eV/Å), and `number_of_atoms`.

In [ ]:
max_num = 2  # match the value used in SimpleUnary

df = pd.read_pickle(f'Unary/{max_num}/everything_training_data.pckl.gz')
print(f"Loaded {len(df)} structures")
df.head()

## Define the ACE basis

We use the simplest possible ACE expansion: a **linear, pair-descriptor-only**
potential (order 1, `lmax=0`).  The `ChebPow` radial basis with 20 functions
and a 6.5 Å cutoff gives a good balance between expressiveness and fitting
speed for this small unary example.

In [ ]:
def make_ace_basis(rmax, number_of_radial_functions, element='Cu'):
    """Minimal linear pair-descriptor ACE basis for a single element."""
    pot_conf = {
        'elements': [element],
        'embeddings': {
            'ALL': {
                'fs_parameters': [1, 1],
                'ndensity': 1,
                'npot': 'FinnisSinclair',
            },
        },
        'bonds': {
            'ALL': {
                'NameOfCutoffFunction': 'cos',
                'dcut': 0.01,
                'radbase': 'ChebPow',
                'radparameters': [2.0],
                'rcut': rmax,
            },
        },
        'functions': {
            'UNARY': {
                'nradmax_by_orders': [number_of_radial_functions],
                'lmax_by_orders':    [0],
            }
        }
    }
    return pyace.create_multispecies_basis_config(pot_conf)


bbasis = make_ace_basis(rmax=6.5, number_of_radial_functions=20, element='Cu')

## Build the design matrix

`LinearACEDataset` projects all training structures onto the ACE basis.  The
resulting design matrix has one row per energy observation plus three rows per
force component.

In [ ]:
ds = plf.LinearACEDataset(bbasis, df)

%time ds.construct_design_matrix()

## Fit

`LinearACEFit` wraps scikit-learn's `Ridge` solver.  For this small dataset
the fit takes milliseconds.

In [ ]:
lf = plf.LinearACEFit(train_dataset=ds)

%time lf.fit()

## Training errors

`compute_errors` returns MAE and RMSE for energies per atom (`epa_*`) and
force components (`f_comp_*`).

In [ ]:
lf.compute_errors()

## Inspect the fitted pair potential

A dimer curve (two Cu atoms at varying separation) is an easy sanity check:
the ACE potential should reproduce the Morse reference minimum near 2.8 Å.

In [ ]:
def dimer_curve(calc, r_values):
    """Evaluate a dimer energy curve for a given ASE calculator.

    Args:
        calc: ASE calculator
        r_values: 1-D array of Cu–Cu distances in Å

    Returns:
        (r_values, energies) — both as numpy arrays
    """
    energies = []
    for r in r_values:
        s = Atoms(['Cu', 'Cu'], positions=[[0, 0, 0], [r, 0, 0]], cell=[40] * 3)
        s.calc = calc
        energies.append(s.get_potential_energy())
    return r_values, np.array(energies)


acef      = pyace.PyACECalculator(lf.get_bbasis())
reference = MorsePotential(epsilon=0.3, r0=2.55265548 * 1.10619396, rho0=4)

r = np.linspace(2.0, 7.0, 200)
_, e_ace = dimer_curve(acef, r)
_, e_ref = dimer_curve(reference, r)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(r, e_ace, label='ACE')
ax1.plot(r, e_ref, label='Morse reference', linestyle='--')
ax1.set_xlabel(r'$r$ [Å]')
ax1.set_ylabel(r'$E$ [eV]')
ax1.legend()
ax1.set_title('Dimer energy')

ax2.plot(r, e_ace - e_ref)
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.set_xlabel(r'$r$ [Å]')
ax2.set_ylabel(r'$\Delta E$ [eV]')
ax2.set_title('Residual (ACE − reference)')

plt.tight_layout()